# Chapter 3 — CPW feedline and readout S21

Engineer course · source candidate · CONVERGING

# Chapter 3 — CPW feedline and readout S21

This complete Chapter is its own clean-kernel execution unit. Its three
web Lessons are reading views over these ordered source fragments; they
are not independent notebooks. QMD is the editable authority, while
`chapter.ipynb` is a generated zero-output transport artifact.

## Lesson 1 — Build a two-Port feedline and readout

### Start from a clean Chapter kernel

Follow the [engineer course setup](../setup.qmd), then open this
Chapter’s `chapter.ipynb`, restart its Python kernel, and use **Run
All**. Chapter 3 is a new execution unit; it does not continue Chapter 1
or 2 state.

The main Plan below is the exact bounded two-Port teaching declaration.
Each line body is a one-section finite-pi approximation to a scalar
lossless CPW model. The 1 mm sections share one feedline tap; a fixed 6
fF capacitor couples that tap directly to a grounded 110 fF/5.8 nH
readout.

In [ ]:
from IPython.display import display

from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    CircuitRun,
    DirectSolveSpec,
    RLGC,
    SParameterTrace,
    Theme,
    components,
    units as u,
)

plan = CircuitPlan(id="two_port_feedline_readout")
feedline = plan.subsystem(id="feedline")
readout = plan.subsystem(id="readout")

cpw = RLGC(
    conductors=("signal",),
    reference_conductor="ground",
    resistance_per_length=[[0.0]] * u.ohm / u.m,
    inductance_per_length=[[420.0]] * u.nH / u.m,
    conductance_per_length=[[0.0]] * u.S / u.m,
    capacitance_per_length=[[175.0]] * u.pF / u.m,
)
left = feedline.add(
    components.transmission_line(
        id="left", length=1.0 * u.mm, rlgc=cpw, n_sections=1
    )
)
right = feedline.add(
    components.transmission_line(
        id="right", length=1.0 * u.mm, rlgc=cpw, n_sections=1
    )
)
feedline_input = feedline.bus(id="input")
feedline_middle = feedline.bus(id="middle")
feedline_output = feedline.bus(id="output")
feedline.series(
    id="left_section",
    start=feedline_input,
    elements=(
        left.between(
            left.pin("head", conductor="signal"),
            left.pin("tail", conductor="signal"),
        ),
    ),
    end=feedline_middle,
)
feedline.series(
    id="right_section",
    start=feedline_middle,
    elements=(
        right.between(
            right.pin("head", conductor="signal"),
            right.pin("tail", conductor="signal"),
        ),
    ),
    end=feedline_output,
)
feedline_input_pin = feedline.expose_pin(id="input", at=feedline_input)
feedline_tap_pin = feedline.expose_pin(id="tap", at=feedline_middle)
feedline_output_pin = feedline.expose_pin(id="output", at=feedline_output)

The reference conductor is RLGC metadata; `input`, `middle`, and
`output` are the three authored signal buses. `head` and `tail` preserve
each body’s terminal ordering and do not infer a direction of physical
power flow.

In [ ]:
readout_capacitor = readout.add(
    components.capacitor(id="capacitor", capacitance=110.0 * u.fF)
)
readout_inductor = readout.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)
readout_bus = readout.bus(id="node")
readout.parallel(
    id="parallel_lc",
    start=readout_bus,
    branches=((readout_capacitor,), (readout_inductor,)),
    end=readout.ground,
)
readout_pin = readout.expose_pin(id="readout_node", at=readout_bus)

The readout’s public Pin is a wiring boundary. This Chapter does not
need a readout analysis Coordinate: its requested observable is the
complete two-Port feedline scattering response.

In [ ]:
input_bus = plan.bus(id="feedline_in")
output_bus = plan.bus(id="feedline_out")
plan.link(id="feedline_input", endpoints=(input_bus, feedline_input_pin))
plan.link(id="feedline_output", endpoints=(output_bus, feedline_output_pin))

coupler = plan.add(
    components.capacitor(
        id="feedline_readout_coupler",
        capacitance=6.0 * u.fF,
    )
)
plan.series(
    id="feedline_readout",
    start=feedline_tap_pin,
    elements=(coupler,),
    end=readout_pin,
)
feedline_in_port = plan.add_port(
    id="feedline_in",
    at=input_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
feedline_out_port = plan.add_port(
    id="feedline_out",
    at=output_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

The two logical Ports own their declared 50 ohm loads. The readout is a
side-coupled internal subsystem, not a third Port.

## Lesson 2 — Review what the fixed diagram can show

### Preserve the complete-Plan layout outcome

Rendering is independent of numerical execution. For this complete Plan,
the current fixed Default authoring layout reports a typed
`schematic_layout` failure. The cell catches only that stage so another
validation error still stops execution.

In [ ]:
from scnsim.errors import SCNSimValidationError

complete_diagram = None
complete_diagram_error = None
try:
    complete_diagram = plan.render_schematic(
        CircuitDiagramSpec(
            representation="authoring",
            theme=Theme.AUTO,
            show_parameter_values=True,
            show_provenance=True,
        )
    )
except SCNSimValidationError as error:
    if error.stage != "schematic_layout":
        raise
    complete_diagram_error = error
    display(
        {
            "complete fixed layout unavailable": str(error),
            "stage": error.stage,
            "evidence": dict(error.evidence),
        }
    )
if complete_diagram is not None:
    display(complete_diagram.show())

> **Current complete-Plan layout limitation**
>
> The exact complete declaration currently reports
> `fixed connector intersects occupied geometry`. Chapter 3 publishes no
> complete-Plan SVG certificate and does not retry, search for another
> placement, or alter the circuit to obtain a drawing.

### Use honest standalone illustrations

`SubsystemPlan` is not independently renderable. The next two Plans are
therefore explicitly separate illustrations with distinct Plan
identities. They repeat the canonical physical values for the feedline
and readout pieces, but they are not projections, substitutes, or
numerical fallbacks for the main Plan.

In [ ]:
feedline_figure_plan = CircuitPlan(id="ch3_feedline_subcircuit_illustration")
figure_cpw = RLGC(
    conductors=("signal",),
    reference_conductor="ground",
    resistance_per_length=[[0.0]] * u.ohm / u.m,
    inductance_per_length=[[420.0]] * u.nH / u.m,
    conductance_per_length=[[0.0]] * u.S / u.m,
    capacitance_per_length=[[175.0]] * u.pF / u.m,
)
figure_left = feedline_figure_plan.add(
    components.transmission_line(
        id="left", length=1.0 * u.mm, rlgc=figure_cpw, n_sections=1
    )
)
figure_right = feedline_figure_plan.add(
    components.transmission_line(
        id="right", length=1.0 * u.mm, rlgc=figure_cpw, n_sections=1
    )
)
figure_input = feedline_figure_plan.bus(id="input")
figure_middle = feedline_figure_plan.bus(id="middle")
figure_output = feedline_figure_plan.bus(id="output")
feedline_figure_plan.series(
    id="left_section",
    start=figure_input,
    elements=(
        figure_left.between(
            figure_left.pin("head", conductor="signal"),
            figure_left.pin("tail", conductor="signal"),
        ),
    ),
    end=figure_middle,
)
feedline_figure_plan.series(
    id="right_section",
    start=figure_middle,
    elements=(
        figure_right.between(
            figure_right.pin("head", conductor="signal"),
            figure_right.pin("tail", conductor="signal"),
        ),
    ),
    end=figure_output,
)
feedline_figure_plan.add_port(
    id="input",
    at=figure_input,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
feedline_figure_plan.add_port(
    id="output",
    at=figure_output,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
feedline_illustration = feedline_figure_plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=False,
    )
)
feedline_illustration.show()

In [ ]:
readout_figure_plan = CircuitPlan(id="ch3_readout_subcircuit_illustration")
figure_readout_capacitor = readout_figure_plan.add(
    components.capacitor(id="capacitor", capacitance=110.0 * u.fF)
)
figure_readout_inductor = readout_figure_plan.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)
figure_readout_bus = readout_figure_plan.bus(id="node")
readout_figure_plan.parallel(
    id="parallel_lc",
    start=figure_readout_bus,
    branches=((figure_readout_capacitor,), (figure_readout_inductor,)),
    end=readout_figure_plan.ground,
)
readout_illustration = readout_figure_plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=False,
    )
)
readout_illustration.show()

The following is only a topology reading aid; it is not a rendered
circuit, electrical evidence, or an ASCDLS certificate:

``` text
feedline_in -- [1 mm CPW] -- tap -- [1 mm CPW] -- feedline_out
                               |
                              6 fF
                               |
                       110 fF || 5.8 nH
                               |
                             ground
```

## Lesson 3 — Request the named two-Port S21 trace

### Declare one finite sample grid and one channel

The selected network is `run.original`: the complete main Plan has
exactly the two terminated feedline Ports needed by this request.
`transmission` names the output-from-input projection of the complete
Direct S matrix; it does not launch a separate solve.

In [ ]:
run = CircuitRun(plan=plan, workspace="workspaces/engineer-chapter-03")
view = run.original
sample_frequencies = [5.5, 6.0, 6.5] * u.GHz
transmission_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(),
    output_port="feedline_out",
    output_mode=(),
)
direct_spec = DirectSolveSpec(
    frequencies=sample_frequencies,
    traces=(transmission_trace,),
)
run.explain(view, direct_spec).show()

The grid contains exactly three declared samples. Lines drawn between
them are presentation only: this lesson does not infer a continuous
resonance, locate a dip, interpolate, or evaluate an undeclared
frequency.

In [ ]:
direct = run.solve(view, direct_spec)
transmission = direct.traces["transmission"]

`transmission.value` retains the complex dimensionless S21 values. The
plot below shows magnitude in dB, while the generated CSV preserves real
and imaginary parts at every declared frequency.

In [ ]:
display(direct.s.view)
display(transmission.frequencies)
display(transmission.value)
transmission.plot(component="magnitude", magnitude="db", theme=Theme.AUTO)

This numerical plot is a presentation of a verified Direct Result, not
an ASCDLS certificate. The two standalone schematics in Lesson 2 have
their own distinct illustrative Plan identities.